In [1]:
import yfinance as yf
import pandas as pd

In [2]:
rb = yf.download("RB=F", start="2015-01-01",
                  auto_adjust=False, multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [3]:
rb

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2015-01-02,1.4334,1.4334,1.5072,1.4287,1.4806,42774
2015-01-05,1.3814,1.3814,1.4404,1.3615,1.4403,46187
2015-01-06,1.3543,1.3543,1.3989,1.3437,1.3750,61161
2015-01-07,1.3376,1.3376,1.3675,1.3110,1.3451,56396
2015-01-08,1.3409,1.3409,1.3529,1.3127,1.3346,62629
...,...,...,...,...,...,...
2026-04-30,3.7715,3.7715,3.8232,3.6846,3.7535,71871
2026-05-01,3.5952,3.5952,3.6768,3.5355,3.6311,63683
2026-05-04,3.7382,3.7382,3.7640,3.5400,3.5690,51258


In [4]:
rb.isnull().sum()

Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

In [5]:
rb = rb[["Close"]].copy()
rb.columns = ["mops_rbob"]
rb = rb.reset_index()
rb.columns = ["date", "mops_rbob"]
rb = rb.set_index("date").sort_index()

In [6]:
weekly_rb = rb.resample("W-TUE").agg(
    mops_rbob_close=("mops_rbob", "last"),
    mops_rbob_mean=("mops_rbob", "mean"),
)

# convert from gallon to barrel
weekly_rb = weekly_rb * 42  # cents/gallon → USD/barrel
# weekly_rb.index = weekly_rb.index + pd.Timedelta(days=3)
# weekly_rb.index.name = "week_monday"

In [7]:
weekly_rb

,mops_rbob_close,mops_rbob_mean
date,,
2015-01-06,56.880601,58.367401
2015-01-13,53.276999,54.975479
2015-01-20,55.137602,55.877851
2015-01-27,56.704202,56.036401
2015-02-03,67.254600,60.982321
...,...,...
2026-04-14,127.659000,127.674960
2026-04-21,134.811600,130.740123
2026-04-28,149.536800,145.605599


In [8]:
weekly_rb.to_parquet("../data/semi-cleaned/mops_rbob_weekly.parquet")

In [9]:
trymoto = pd.read_parquet('../data/semi-cleaned/mops_rbob_weekly.parquet')
trymoto

,mops_rbob_close,mops_rbob_mean
date,,
2015-01-06,56.880601,58.367401
2015-01-13,53.276999,54.975479
2015-01-20,55.137602,55.877851
2015-01-27,56.704202,56.036401
2015-02-03,67.254600,60.982321
...,...,...
2026-04-14,127.659000,127.674960
2026-04-21,134.811600,130.740123
2026-04-28,149.536800,145.605599
